# Create IADB Science and Technology Awards

Creates OpenAlex award rows from the Inter-American Development Bank's official Open Data project dataset, filtered to sector code `ST` / `SCIENCE AND TECHNOLOGY`.

**Prerequisites:**
- Admin/human with AWS credentials runs `scripts/local/iadb_scitech_to_s3.py` to download the current CKAN rows, write parquet, and upload it.
- Contractor has prepared the script, notebook, registry entry, tracker row, and local QA, but does not have S3/Databricks access.
- The downloader writes parquet columns as strings per `plans/awards/how-to-add-a-funder.md`; this notebook does type conversion with `TRY_CAST` / `TRY_TO_DATE`.

**Data source:** Official IDB Open Data CKAN dataset at `https://data.iadb.org/dataset/idb-projects-dataset`, discovered via `package_show` and fetched with `datastore_search` from the English resource `814b7b54-477a-4c25-b3bf-6be05412069d`.

**Source scope:** Sector code `ST` (`SCIENCE AND TECHNOLOGY`) only. Local full `--skip-upload` validation on 2026-05-27 fetched 757 project rows, 1966-2026, with 100% native project ID, title, objective, amount, and currency coverage. IDB publishes the amount field as original approved USEQ amount; this ingest maps it to USD-equivalent `amount` with `currency = 'USD'`.

**S3 input:** `s3a://openalex-ingest/awards/iadb_scitech/iadb_scitech_projects.parquet`

**Funder:** Inter-American Development Bank, `F4320307862`, DOI `10.13039/100004429`, no ROR in the OpenAlex API response.

**Provenance:** `iadb_project_search_scitech`


## Step 1: Create Raw Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.iadb_scitech_raw
USING delta
AS
SELECT
    *,
    current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/iadb_scitech/iadb_scitech_projects.parquet`;


In [ ]:
%sql
-- Check raw row count. Local validation on 2026-05-27 found 757 rows.
SELECT COUNT(*) as total_iadb_scitech_projects
FROM openalex.awards.iadb_scitech_raw;


In [ ]:
%sql
-- Verify actual column names before transform logic references them.
DESCRIBE openalex.awards.iadb_scitech_raw;


In [ ]:
%sql
-- Sample raw IADB science and technology data.
SELECT
    funder_award_id,
    display_name,
    country_name,
    status,
    operation_type,
    subsector_name,
    amount,
    currency,
    approved_date,
    signed_date,
    landing_page_url
FROM openalex.awards.iadb_scitech_raw
LIMIT 10;


## Step 1.5: Inspect Raw Data, Money, Dates, and Native Keys

In [ ]:
%sql
-- Money-field scan per runbook Step 1.5. Uses information_schema, not DESCRIBE as a subquery.
SELECT column_name
FROM openalex.information_schema.columns
WHERE table_schema = 'awards'
  AND table_name = 'iadb_scitech_raw'
  AND LOWER(column_name) RLIKE
    'amount|amt|total|value|sum|funded|funding|cost|budget|grant|award|valor|currency|counterpart';


In [ ]:
%sql
-- Native ID uniqueness check. Should be 757 total and 757 distinct in the full run.
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids,
    SUM(CASE WHEN funder_award_id IS NULL OR TRIM(funder_award_id) = '' THEN 1 ELSE 0 END) AS missing_funder_award_id
FROM openalex.awards.iadb_scitech_raw;


In [ ]:
%sql
-- Amount and currency coverage. IDB's source field is original approved USEQ amount, mapped to USD-equivalent.
SELECT
    COUNT(*) AS total_rows,
    COUNT(amount) AS rows_with_amount,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_with_amount,
    COUNT(currency) AS rows_with_currency,
    collect_set(currency) AS currencies,
    ROUND(MIN(TRY_CAST(amount AS DOUBLE)), 2) AS min_amount,
    ROUND(MAX(TRY_CAST(amount AS DOUBLE)), 2) AS max_amount,
    ROUND(AVG(TRY_CAST(amount AS DOUBLE)), 2) AS avg_amount,
    ROUND(SUM(TRY_CAST(amount AS DOUBLE)), 2) AS total_amount
FROM openalex.awards.iadb_scitech_raw;


In [ ]:
%sql
-- Approval/signature date and year coverage.
SELECT
    COUNT(*) AS total_rows,
    COUNT(approved_date) AS rows_with_approved_date,
    ROUND(try_divide(COUNT(approved_date), COUNT(*)) * 100.0, 1) AS pct_approved_date,
    COUNT(signed_date) AS rows_with_signed_date,
    ROUND(try_divide(COUNT(signed_date), COUNT(*)) * 100.0, 1) AS pct_signed_date,
    MIN(TRY_CAST(source_year AS INT)) AS min_year,
    MAX(TRY_CAST(source_year AS INT)) AS max_year
FROM openalex.awards.iadb_scitech_raw;


In [ ]:
%sql
-- Scheme / operation-type distribution.
SELECT
    operation_type,
    funding_type,
    COUNT(*) AS rows,
    ROUND(SUM(TRY_CAST(amount AS DOUBLE)), 2) AS total_amount
FROM openalex.awards.iadb_scitech_raw
GROUP BY operation_type, funding_type
ORDER BY rows DESC, operation_type;


In [ ]:
%sql
-- Status and country distribution.
SELECT status, country_name, COUNT(*) AS rows
FROM openalex.awards.iadb_scitech_raw
GROUP BY status, country_name
ORDER BY rows DESC, status, country_name
LIMIT 40;


## Step 1.6: Funder Existence Fail-Fast

The transform in Step 2 does `CROSS JOIN openalex.common.funder WHERE funder_id = 4320307862`. If this returns zero rows, STOP; the insert would silently emit no awards.


In [ ]:
%sql
-- Must return exactly 1 row for Inter-American Development Bank.
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.common.funder
WHERE funder_id = 4320307862;


In [ ]:
%sql
-- Fails the cell if the Crossref-registered funder row is missing from openalex.common.funder.
SELECT CASE
    WHEN COUNT(*) = 1 THEN 'OK: IADB funder row exists'
    ELSE raise_error('IADB funder row missing from openalex.common.funder')
END AS funder_check
FROM openalex.common.funder
WHERE funder_id = 4320307862;


## Step 2: Transform to OpenAlex Award Schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.iadb_scitech_awards
USING delta
AS
WITH
iadb_funder AS (
    SELECT
        funder_id,
        display_name,
        ror_id,
        doi
    FROM openalex.common.funder
    WHERE funder_id = 4320307862
),

raw_prepared AS (
    SELECT
        *,
        LOWER(TRIM(funder_award_id)) AS native_award_id,
        TRY_CAST(amount AS DOUBLE) AS parsed_amount,
        CASE WHEN TRY_CAST(amount AS DOUBLE) IS NOT NULL THEN 'USD' ELSE NULL END AS parsed_currency,
        COALESCE(
            TRY_TO_DATE(approved_date, 'yyyy-MM-dd'),
            TRY_TO_DATE(signed_date, 'yyyy-MM-dd')
        ) AS parsed_start_date,
        TRY_CAST(source_year AS INT) AS parsed_start_year
    FROM openalex.awards.iadb_scitech_raw
    WHERE funder_award_id IS NOT NULL
      AND TRIM(funder_award_id) != ''
      AND display_name IS NOT NULL
      AND TRIM(display_name) != ''
),

awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(f.funder_id, ':', g.native_award_id))) % 9000000000 as id,
        TRIM(g.display_name) as display_name,
        NULLIF(TRIM(g.description), '') as description,
        f.funder_id,
        g.native_award_id as funder_award_id,
        g.parsed_amount as amount,
        g.parsed_currency as currency,
        struct(
            CONCAT('https://openalex.org/F', f.funder_id) as id,
            f.display_name,
            f.ror_id,
            f.doi
        ) as funder,
        COALESCE(NULLIF(TRIM(g.funding_type), ''), 'research') as funding_type,
        COALESCE(NULLIF(TRIM(g.subsector_name), ''), NULLIF(TRIM(g.sector_name), ''), 'SCIENCE AND TECHNOLOGY') as funder_scheme,
        'iadb_project_search_scitech' as provenance,
        g.parsed_start_date as start_date,
        CAST(NULL AS DATE) as end_date,
        COALESCE(YEAR(g.parsed_start_date), g.parsed_start_year) as start_year,
        CAST(NULL AS INT) as end_year,
        CAST(NULL AS STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) as lead_investigator,
        CAST(NULL AS STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) as co_lead_investigator,
        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >>) as investigators,
        NULLIF(TRIM(g.landing_page_url), '') as landing_page_url,
        CAST(NULL AS STRING) as doi,
        CONCAT('https://api.openalex.org/works?filter=awards.id:G',
               CAST(abs(xxhash64(CONCAT(f.funder_id, ':', g.native_award_id))) % 9000000000 AS STRING)) as works_api_url,
        current_timestamp() as created_date,
        current_timestamp() as updated_date
    FROM raw_prepared g
    CROSS JOIN iadb_funder f
)
SELECT *
FROM awards_transformed;


## Step 3: Delete Previous Source Rows and Insert Priority 156

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'iadb_project_search_scitech' AND priority = 156;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id,
    display_name,
    description,
    funder_id,
    funder_award_id,
    amount,
    currency,
    funder,
    funding_type,
    funder_scheme,
    provenance,
    start_date,
    end_date,
    start_year,
    end_year,
    lead_investigator,
    co_lead_investigator,
    investigators,
    landing_page_url,
    doi,
    works_api_url,
    created_date,
    updated_date,
    156 as priority
FROM openalex.awards.iadb_scitech_awards;


## Step 6: Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_iadb_scitech_awards
FROM openalex.awards.iadb_scitech_awards;


In [ ]:
%sql
DESCRIBE openalex.awards.iadb_scitech_awards;


In [ ]:
%sql
SELECT
    id,
    display_name,
    funder_award_id,
    funder_scheme,
    funding_type,
    amount,
    currency,
    start_date,
    landing_page_url
FROM openalex.awards.iadb_scitech_awards
LIMIT 10;


In [ ]:
%sql
SELECT COUNT(*) AS total_rows, COUNT(DISTINCT id) AS distinct_openalex_ids, COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids
FROM openalex.awards.iadb_scitech_awards;


In [ ]:
%sql
SELECT funder.display_name, funder_id, COUNT(*) as cnt
FROM openalex.awards.iadb_scitech_awards
GROUP BY funder.display_name, funder_id
ORDER BY cnt DESC;


In [ ]:
%sql
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(description) as has_description,
    COUNT(amount) as has_amount,
    COUNT(currency) as has_currency,
    COUNT(start_date) as has_start_date,
    COUNT(landing_page_url) as has_url,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) as pct_with_amount,
    ROUND(try_divide(COUNT(description), COUNT(*)) * 100.0, 1) as pct_description,
    ROUND(try_divide(COUNT(start_date), COUNT(*)) * 100.0, 1) as pct_start_date,
    ROUND(SUM(amount), 2) as total_funding_usd_equiv
FROM openalex.awards.iadb_scitech_awards;


In [ ]:
%sql
SELECT
    COUNT(*) AS total,
    SUM(CASE WHEN amount > 0 THEN 1 ELSE 0 END) AS rows_with_positive_amount,
    ROUND(try_divide(SUM(CASE WHEN amount > 0 THEN 1 ELSE 0 END), COUNT(*)) * 100.0, 1) AS pct_positive_amount,
    COUNT(DISTINCT currency) AS distinct_currencies,
    collect_set(currency) AS currencies,
    ROUND(MIN(amount), 2) AS min_amount,
    ROUND(MAX(amount), 2) AS max_amount,
    ROUND(AVG(amount), 2) AS avg_amount,
    ROUND(SUM(amount), 2) AS total_amount
FROM openalex.awards.iadb_scitech_awards;


In [ ]:
%sql
SELECT funding_type, funder_scheme, COUNT(*) AS rows, ROUND(SUM(amount), 2) AS total_amount
FROM openalex.awards.iadb_scitech_awards
GROUP BY funding_type, funder_scheme
ORDER BY rows DESC, funding_type, funder_scheme
LIMIT 40;


In [ ]:
%sql
-- Verify the combined raw table received exactly this source/priority after Step 3.
SELECT provenance, priority, COUNT(*) AS rows
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'iadb_project_search_scitech'
  AND priority = 156
GROUP BY provenance, priority;


In [ ]:
%sql
-- Native-code uniqueness check; must return zero rows.
SELECT funder_award_id, COUNT(*) AS rows
FROM openalex.awards.iadb_scitech_awards
GROUP BY funder_award_id
HAVING COUNT(*) > 1;


## Handoff / Admin Notes

- Contractor local validation produced 757 rows from the official IDB Open Data API and wrote string-typed parquet locally.
- Contractor has no AWS/S3/Databricks access. Admin must run `scripts/local/iadb_scitech_to_s3.py` without `--skip-upload`, run this notebook on Databricks, inspect Step 6, and then run the combined `CreateAwards.ipynb` after QA.
- Do not add job YAML until the admin upload, notebook run, and QA are complete.
